# HGP-clusterer : 3D puis 4D Panoptic Segmentation sur SemanticKITTI

Ce notebook implémente un pipeline de segmentation panoptique en "streaming" en utilisant **HGP-clusterer**.

**Pipeline :**
1.  **Setup** : Installation des dépendances.
2.  **Data** : Chargement d'une séquence SemanticKITTI.
3.  **Preprocessing** : Construction des nuages de points.
4.  **Clustering & Tracking (Streaming)** : 
    - Initialisation (Frame 0) : Segmentation d'instance 3D (ou BEV) et création des représentations des clusters (volume, classe, vitesse).
    - Suivi (Frames suivantes) : *En cours (itération frame par frame)*.
5.  **Evaluation** : *Adaptée pour le suivi*.
6.  **Visualisation** : Rendu interactif.

In [ ]:
# @title 1.1 Choix du Backend Géométrique
# 'geogram' est recommandé pour la vitesse (headless). 'cgal' est plus lent mais exact.
BACKEND = 'geogram'  # @param ['geogram', 'cgal']
print(f"Backend sélectionné : {BACKEND}")

In [ ]:
# @title 1.2 Installation des dépendances système
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git libeigen3-dev libomp-dev

if BACKEND == 'cgal':
    # libboost-all-dev est souvent nécessaire pour que CMake détecte correctement CGAL
    !apt-get install -y -qq libcgal-dev libtbb-dev libtbbmalloc2 libgmp-dev libmpfr-dev libboost-all-dev

In [ ]:
# @title 1.3 Installation des dépendances Python
!pip install -q --upgrade pip setuptools wheel Cython cmake jedi gdown pybind11
!pip install -q numpy scipy scikit-learn plotly tqdm joblib open3d plyfile hdbscan pandas matplotlib pyyaml

In [ ]:
%%bash
# @title 1.4 Installation de HGP-clusterer et SemanticKITTI-API
set -euo pipefail
WORKDIR="/content"
mkdir -p "${WORKDIR}"
cd "${WORKDIR}"

# HGP-clusterer
if [ -d HGP-clusterer ]; then
    git -C HGP-clusterer pull --ff-only
else
    git clone https://github.com/Ludwig-H/HGP-clusterer.git
fi

# SemanticKITTI API (pour l'évaluation)
if [ -d semantic-kitti-api ]; then
    git -C semantic-kitti-api pull --ff-only
else
    git clone https://github.com/PRBonn/semantic-kitti-api.git
fi

In [ ]:
# @title 1.5 Compilation de HGP
import os
import sys
import subprocess

WORKDIR = "/content"
os.chdir(WORKDIR)

if BACKEND == 'geogram':
    if not os.path.exists('geogram'):
        print("Clonage de Geogram...")
        !git clone --recursive https://github.com/BrunoLevy/geogram.git
    
    print("Compilation de Geogram (Headless)...")
    !cmake -S geogram -B geogram/build -DCMAKE_BUILD_TYPE=Release -DGEOGRAM_WITH_GRAPHICS=OFF -DGEOGRAM_WITH_LUA=OFF -DGEOGRAM_WITH_GARGANTUA=OFF
    !cmake --build geogram/build --config Release --parallel 4
    !cmake --install geogram/build --prefix /usr/local
    os.environ['GEOGRAM_INSTALL_PREFIX'] = '/usr/local'

elif BACKEND == 'cgal':
    print("Configuration CGAL...")
    
    # -- FIX: Add CGAL path to environment for setup_cgal.py --
    cgal_prefix = "/usr/lib/x86_64-linux-gnu/cmake/CGAL"
    current_cpp = os.environ.get("CMAKE_PREFIX_PATH", "")
    os.environ["CMAKE_PREFIX_PATH"] = f"{current_cpp}:{cgal_prefix}" if current_cpp else cgal_prefix
    # ---------------------------------------------------------

    # On tente de construire l'outil CGAL, mais on continue même en cas d'erreur
    # car le setup.py principal pourrait réussir autrement.
    try:
        subprocess.run(["python3", f"{WORKDIR}/HGP-clusterer/scripts/setup_cgal.py"], check=True)
    except subprocess.CalledProcessError:
        print("⚠️ Attention: Echec du script setup_cgal.py. Tentative de continuation avec le build principal...")

os.chdir(f"{WORKDIR}/HGP-clusterer")
!rm -rf build dist *.egg-info

install_cmd = "pip install --no-build-isolation -v --no-deps ."
if BACKEND == 'geogram':
    install_cmd = f"GEOGRAM_INSTALL_PREFIX=/usr/local {install_cmd}"
elif BACKEND == 'cgal':
    # Ajout du chemin système pour CGAL (Debian/Ubuntu/Colab)
    # Note: On le passe aussi explicitement ici pour être sûr
    install_cmd = f"CGALDELAUNAY_ROOT={WORKDIR}/HGP-clusterer/CGALDelaunay CMAKE_PREFIX_PATH={WORKDIR}/HGP-clusterer:{cgal_prefix} {install_cmd}"

print(f"Exécution : {install_cmd}")
!{install_cmd}

os.environ["CGALDELAUNAY_ROOT"] = f"{WORKDIR}/HGP-clusterer/CGALDelaunay"

try:
    from hgp_clusterer import HGPClusterer
    print("✅ HGPClusterer installé.")
except ImportError as e:
    print(f"❌ Erreur import HGP: {e}")

In [ ]:
# @title 2.1 Configuration Séquence et Téléchargement
# IMPORTANT : Si vous ne voulez tester qu'une seule séquence, lancez cette cellule.
# Le téléchargement via gdown --folder récupère tout le dossier si on ne filtre pas.
# Ici, on télécharge tout le dataset SemanticKITTI (partiel) fourni via le lien Drive.

SEQUENCE_TO_TEST = 8 # @param {type:"integer"}
DOWNLOAD_DATA = True # @param {type:"boolean"}

# Choix du mode de téléchargement :
# - 'Folder' : Télécharge fichier par fichier (Très lent pour 10k fichiers, mais utile si on a que le lien du dossier)
# - 'Zip' : Télécharge une archive unique et décompresse (Beaucoup plus rapide, recommandé)
DOWNLOAD_MODE = "Zip" # @param ["Folder", "Zip"]

# IDs Google Drive par séquence
# Remplissez ce dictionnaire avec les IDs des dossiers ou des zips pour chaque séquence.
SEQUENCE_DRIVE_IDS = {
    8: {
        "Folder": "1UqFKvekjyic6L_8KD1kcv8MuGmQMIk0A",
        "Zip": "1ZoZtzdFAkPWYHT8sFsHjwyEmFHbpaIQH"
    }
}

# Dossier Racine (Fallback si ID spécifique non trouvé en mode Folder)
ROOT_FOLDER_ID = "1ORVzSo-TWbNHeAC0-k3mxX9AiJHI_tVu"

if DOWNLOAD_DATA:
    import os
    import shutil
    
    # Destination racine
    base_dest = "/content/semantic_kitti_data"
    seq_str = f"{SEQUENCE_TO_TEST:02d}"
    target_dir = os.path.join(base_dest, seq_str)
    
    if not os.path.exists(target_dir):
        print(f"Démarrage du téléchargement (Mode : {DOWNLOAD_MODE})...")
        
        # Récupération des IDs pour la séquence choisie
        seq_ids = SEQUENCE_DRIVE_IDS.get(SEQUENCE_TO_TEST, {})
        
        if DOWNLOAD_MODE == "Zip":
            zip_id = seq_ids.get("Zip")
            if not zip_id:
                print(f"⚠️ Aucun ID Zip trouvé pour la séquence {SEQUENCE_TO_TEST}. Veuillez remplir SEQUENCE_DRIVE_IDS.")
                print("Passage automatique en mode Folder (Fallback)...")
                # On ne lance pas d'erreur, on essaie le folder si possible, sinon root
                DOWNLOAD_MODE = "Folder" 
            else:
                print(f"Téléchargement de l'archive Zip (ID: {zip_id})...")
                zip_path = os.path.join(base_dest, "sequence.zip")
                os.makedirs(base_dest, exist_ok=True)
                
                # Téléchargement
                !gdown {zip_id} -O {zip_path} --quiet
                
                print("Décompression...")
                # On décompresse
                !unzip -q {zip_path} -d {target_dir}
                !rm {zip_path}
                
                # Vérification de la structure (si le zip contenait un sous-dossier, on remonte)
                if not os.path.exists(os.path.join(target_dir, "velodyne")):
                    # Tentative de correction automatique
                    sub_dirs = [d for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))]
                    if len(sub_dirs) == 1:
                        inner_dir = os.path.join(target_dir, sub_dirs[0])
                        print(f"Structure imbriquée détectée, déplacement de {inner_dir} vers {target_dir}...")
                        for item in os.listdir(inner_dir):
                            shutil.move(os.path.join(inner_dir, item), target_dir)
                        os.rmdir(inner_dir)

        # Note: Ce bloc est exécuté si mode Folder OU si fallback depuis Zip
        if DOWNLOAD_MODE == "Folder":
            folder_id = seq_ids.get("Folder")
            if not folder_id:
                # Fallback sur le root folder (pas idéal mais fonctionnel)
                print(f"ID spécifique Folder manquant pour la séquence {SEQUENCE_TO_TEST}.")
                print(f"Tentative de téléchargement via le dossier racine {ROOT_FOLDER_ID}...")
                !gdown --folder {ROOT_FOLDER_ID} -O {base_dest} --quiet --remaining-ok
            else:
                print(f"Téléchargement du dossier (ID: {folder_id})...")
                # On crée le dossier de la séquence pour gdown
                !gdown --folder {folder_id} -O {target_dir} --quiet --remaining-ok
        
        print("Téléchargement terminé.")
    else:
        print(f"Dossier {target_dir} existe déjà. Skip download.")
else:
    print("Téléchargement désactivé.")

print(f"Séquence cible pour le test : {SEQUENCE_TO_TEST}")

In [ ]:
# @title 2.2 Loader SemanticKITTI
import os
import numpy as np
import glob

class SemanticKITTILoader:
    def __init__(self, base_path, sequence_num):
        self.seq_str = f"{sequence_num:02d}"

        # Recherche du dossier de la séquence.
        # Structure attendue : base_path/08 ou base_path/sequences/08

        # 1. Chercher direct
        possible_paths = glob.glob(f"{base_path}/{self.seq_str}")

        # 2. Chercher dans un sous-dossier 'sequences' (structure officielle KITTI)
        if not possible_paths:
            possible_paths = glob.glob(f"{base_path}/**/sequences/{self.seq_str}", recursive=True)

        # 3. Chercher récursivement n'importe où (au cas où gdown a créé une structure intermédiaire)
        if not possible_paths:
             possible_paths = glob.glob(f"{base_path}/**/{self.seq_str}", recursive=True)

        # Filtrer pour ne garder que les vrais dossiers contenant 'velodyne'
        valid_paths = []
        for p in possible_paths:
            if os.path.exists(os.path.join(p, 'velodyne')):
                valid_paths.append(p)

        if not valid_paths:
            raise ValueError(f"Séquence {self.seq_str} introuvable dans {base_path}. Vérifiez que le dossier 'velodyne' est bien présent.")

        self.seq_path = valid_paths[0]
        print(f"Séquence chargée : {self.seq_path}")

        self.velo_path = os.path.join(self.seq_path, 'velodyne')
        self.label_path = os.path.join(self.seq_path, 'labels')
        self.poses_file = os.path.join(self.seq_path, 'poses.txt')
        self.calib_file = os.path.join(self.seq_path, 'calib.txt')

        # Fallback pour poses.txt/calib.txt s'ils sont dans le dossier parent (structure dataset/sequences/08)
        if not os.path.exists(self.poses_file):
             # Essayer de remonter d'un niveau (dataset/sequences/) ou deux
             parent = os.path.dirname(self.seq_path) # dataset/sequences
             grandparent = os.path.dirname(parent) # dataset

             # Cas dataset/poses.txt (peu probable mais...)
             # Cas dataset/sequences/08/poses.txt (standard)
             pass

        self.scan_files = sorted(glob.glob(os.path.join(self.velo_path, '*.bin')))
        self.label_files = sorted(glob.glob(os.path.join(self.label_path, '*.label')))
        self.poses = self._load_poses()
        self.calib = self._load_calib()

    def _load_poses(self):
        if not os.path.exists(self.poses_file):
            print(f"Info: poses.txt non trouvé ({self.poses_file}).")
            return []
        poses = []
        with open(self.poses_file, 'r') as f:
            for line in f:
                values = [float(v) for v in line.strip().split()]
                pose = np.vstack([np.array(values).reshape(3, 4), [0, 0, 0, 1]])
                poses.append(pose)
        return poses

    def _load_calib(self):
        if not os.path.exists(self.calib_file):
            print(f"Info: calib.txt non trouvé ({self.calib_file}).")
            return np.eye(4)
        calib = {}
        with open(self.calib_file, 'r') as f:
            for line in f:
                if ':' not in line: continue
                key, val = line.split(':', 1)
                calib[key] = np.array([float(x) for x in val.split()]).reshape(3, 4)
        if 'Tr' in calib:
            return np.vstack([calib['Tr'], [0, 0, 0, 1]])
        return np.eye(4)

    def get_scan(self, idx, apply_pose=True):
        scan = np.fromfile(self.scan_files[idx], dtype=np.float32).reshape(-1, 4)
        points = scan[:, :3]
        if apply_pose and self.poses and idx < len(self.poses):
            T = self.poses[idx] @ self.calib
            points = (T @ np.hstack([points, np.ones((len(points), 1))]).T).T[:, :3]
        return points

    def get_labels(self, idx):
        if idx >= len(self.label_files): return None, None
        label = np.fromfile(self.label_files[idx], dtype=np.uint32)
        return label & 0xFFFF, label >> 16

    def __len__(self): return len(self.scan_files)

In [ ]:
# @title 3.1 Construction du Nuage 4D
import numpy as np

START_FRAME = 0 # @param {type:"integer"}
NUM_FRAMES = 10 # @param {type:"integer"} (-1 pour toutes les frames)
DT_SCALE = 0.5  # @param {type:"number"}
APPLY_BEV = True # @param {type:"boolean"}
# Mode Sémantique :
# - 'Oracle' : Utilise la vérité terrain fournie par SemanticKITTI pour filtrer les objets mobiles (Things).
# - 'None' : Ne filtre rien (Lance le clustering sur absolument toute la scène, lent et non recommandé).
# (Note: Le chargement d'une prédiction réseau externe viendrait ici dans une future mise à jour)
SEMANTIC_MODE = "Oracle" # @param ["Oracle", "None"]

# Mapping officiel SemanticKITTI (fusionne les classes "moving" avec leur équivalent statique)
LEARNING_MAP = {
  0: 0, 1: 0, 10: 1, 11: 2, 13: 5, 15: 3, 16: 5, 18: 4, 20: 5, 30: 6, 
  31: 7, 32: 8, 40: 9, 44: 10, 48: 11, 49: 12, 50: 13, 51: 14, 52: 0, 
  60: 9, 70: 15, 71: 16, 72: 17, 80: 18, 81: 19, 99: 0, 252: 1, 
  253: 7, 254: 6, 255: 8, 256: 5, 257: 5, 258: 4, 259: 5
}
# Vecteur de mapping rapide (taille max 260)
max_key = max(LEARNING_MAP.keys())
LABEL_MAP_ARRAY = np.zeros(max_key + 1, dtype=np.uint32)
for k, v in LEARNING_MAP.items():
    LABEL_MAP_ARRAY[k] = v

# SemanticKITTI classes "things" dans l'espace mappé (1 à 8)
# 1:car, 2:bicycle, 3:motorcycle, 4:truck, 5:other-vehicle, 6:person, 7:bicyclist, 8:motorcyclist
THINGS_CLASSES = set([1, 2, 3, 4, 5, 6, 7, 8])

# Initialisation des variables pour éviter les NameError
X_clustering = None
X_4d = None
Y_sem = None
Y_inst = None
Time_idx = None
Original_Indices = None # Pour garder la trace si on filtre

try:
    loader = SemanticKITTILoader("/content/semantic_kitti_data", SEQUENCE_TO_TEST)
    points_4d, gt_sem, gt_inst, times, indices = [], [], [], [], []
    
    total_points = 0
    actual_num_frames = len(loader) - START_FRAME if NUM_FRAMES == -1 else NUM_FRAMES
    print(f"Chargement frames {START_FRAME} -> {START_FRAME + actual_num_frames}...")
    for i in range(actual_num_frames):
        idx = START_FRAME + i
        if idx >= len(loader): break

        pts = loader.get_scan(idx, apply_pose=True)
        s_raw, inst = loader.get_labels(idx)
        
        # Mapping des labels sémantiques bruts vers l'espace d'évaluation
        s = LABEL_MAP_ARRAY[s_raw]

        # 4D Point: x, y, z, t
        t_col = np.full((len(pts), 1), i * DT_SCALE)
        pts_4d = np.hstack([pts, t_col])
        
        # Filtre Sémantique
        if SEMANTIC_MODE == "Oracle":
            # On ne garde que les classes "Things"
            mask = np.array([sem in THINGS_CLASSES for sem in s])
            pts_4d = pts_4d[mask]
            s = s[mask]
            inst = inst[mask]
            
            # Si on veut garder l'index original par rapport à la frame (pour de la visulaisation par ex)
            frame_indices = np.arange(len(mask))[mask]
        else:
            frame_indices = np.arange(len(pts))

        points_4d.append(pts_4d)
        gt_sem.append(s)
        gt_inst.append(inst)
        times.extend([i] * len(pts_4d))
        indices.append(frame_indices + total_points)
        total_points += len(pts) # On ajoute le total brut pour les indices absolus

    if points_4d:
        X_4d = np.vstack(points_4d)
        Y_sem = np.hstack(gt_sem)
        Y_inst = np.hstack(gt_inst)
        Time_idx = np.array(times)
        Original_Indices = np.hstack(indices)

        # Bird's Eye View : on utilise uniquement x, y, t pour le clustering
        if APPLY_BEV:
            print(f"Mode Bird's-Eye-View (BEV) activé : Clustering sur (x, y, t).")
            X_clustering = np.column_stack([X_4d[:, 0], X_4d[:, 1], X_4d[:, 3]])
        else:
            print("Mode 4D Complet activé : Clustering sur (x, y, z, t).")
            X_clustering = X_4d

        print(f"Sémantique : Mode {SEMANTIC_MODE}.")
        print(f"Nuage 4D filtré: {X_4d.shape} points conservés.")
        print(f"Input Clustering: {X_clustering.shape}")
    else:
        print("Aucun point chargé ou aucun point 'thing' trouvé. Vérifiez les chemins.")

except Exception as e:
    print(f"Erreur lors du chargement des données: {e}")
    print("---------------------------------------------------------")
    print("⚠️ GÉNÉRATION DE DONNÉES SYNTHÉTIQUES (FALLBACK) ⚠️")
    print("---------------------------------------------------------")
    from sklearn.datasets import make_blobs
    n_samples = 5000
    X_syn, y_syn = make_blobs(n_samples=n_samples, n_features=3, centers=5, cluster_std=1.0)
    synth_frames = NUM_FRAMES if NUM_FRAMES != -1 else 10
    t_syn = np.random.randint(0, synth_frames, size=n_samples) * DT_SCALE
    X_4d = np.column_stack([X_syn, t_syn])
    Y_sem = np.zeros(n_samples, dtype=int)
    Y_inst = y_syn + 1 
    Time_idx = (t_syn / DT_SCALE).astype(int)
    X_clustering = X_4d if not APPLY_BEV else X_4d[:, [0, 1, 3]]
    SEMANTIC_MODE = "None"
    print(f"Données synthétiques générées: {X_clustering.shape}")

In [ ]:
# @title 4.1 HGP Clustering (Streaming Frame 0)
import time
import numpy as np
import os

# --- Import sécurisé de HGPClusterer ---
try:
    from hgp_clusterer import HGPClusterer
except ImportError:
    print("⚠️ Module HGPClusterer introuvable. Tentative de correction du path...")
    import sys
    if "/content/HGP-clusterer/src" not in sys.path:
        sys.path.append("/content/HGP-clusterer/src")
    try:
        from hgp_clusterer import HGPClusterer
        print("✅ HGPClusterer importé avec succès après correction du path.")
    except ImportError as e:
        raise RuntimeError(f"❌ Impossible d'importer HGPClusterer même après correction. Erreur: {e}. Veuillez vérifier la compilation en section 1.5.")

K = 3 # @param {type:"integer"}
MIN_CLUSTER_SIZE = 1 # @param {type:"integer"}
EXP_Z = 1 # @param {type:"number"}

CURRENT_GT_INSTANCES = None

def oracle_Gini(parent_pts_idx, children_pts_idx_list, ε_Gini=0.001):
    """
    Splitting basé sur l'index de Gini (Oracle).
    Utilise CURRENT_GT_INSTANCES qui doit être mis à jour avant le fit.
    """
    global CURRENT_GT_INSTANCES
    if CURRENT_GT_INSTANCES is None or len(parent_pts_idx) == 0: return False
    
    def get_gini(indices):
        if len(indices) == 0: return 0.0
        labels = CURRENT_GT_INSTANCES[indices]
        if len(labels) == 0: return 0.0
        _, counts = np.unique(labels, return_counts=True)
        probs = counts / len(labels)
        return 1.0 - np.sum(probs**2)
    
    gini_p = get_gini(parent_pts_idx)
    if gini_p < ε_Gini: return False
    
    n_total = len(parent_pts_idx)
    gini_c_weighted = 0.0
    for c_pts in children_pts_idx_list:
        if len(c_pts) == 0: continue
        gini_c_weighted += (len(c_pts) / n_total) * get_gini(c_pts)
    
    return gini_c_weighted < (gini_p - ε_Gini)

SPLITTING_REGISTRY = {
    "None": None,
    "oracle_Gini": oracle_Gini
}

SPLITTING_MODE = "None" # @param ["None", "oracle_Gini"] {allow-input: true}
DBSCAN_FACTOR = 0.5 * 1.5 # @param {type:"number"}

# Vérification préalable des données
if X_clustering is None or len(X_clustering) == 0:
    raise RuntimeError("Erreur critique: X_clustering est vide ou n'est pas défini. Veuillez vérifier la cellule 'Construction du Nuage 4D'.")

# Initialisation du tableau global des prédictions (ID des instances)
labels_pred = np.full(len(X_clustering), -1, dtype=int)
global_instance_offset = 0


# --- Priors de tailles d'objets (Inspirés d'Alpine + 3D) ---
# Bounding Boxes 3D (Length, Width, Height) en mètres.
ALPINE_PRIORS = {
    1: {"name": "car",           "L": 4.5,  "W": 1.85, "H": 1.6},
    2: {"name": "bicycle",       "L": 1.8,  "W": 0.6,  "H": 1.1},
    3: {"name": "motorcycle",    "L": 2.2,  "W": 0.9,  "H": 1.3},
    4: {"name": "truck",         "L": 10.0, "W": 2.6,  "H": 3.5},
    5: {"name": "other-vehicle", "L": 12.0, "W": 2.6,  "H": 3.5},
    6: {"name": "person",        "L": 0.6,  "W": 0.6,  "H": 1.75},
    7: {"name": "bicyclist",     "L": 1.8,  "W": 0.75, "H": 1.8},
    8: {"name": "motorcyclist",  "L": 2.2,  "W": 0.9,  "H": 1.8},
}

frames = np.unique(Time_idx)
print(f"Frames à traiter en streaming : {frames}")

# Stockage des informations sur les clusters
clusters_info = {}

for t in frames:
    print(f"\n=== Traitement de la frame {t} ===")
    frame_mask = (Time_idx == t)
    X_frame = X_clustering[frame_mask]
    Y_sem_frame = Y_sem[frame_mask]
    Y_inst_frame = Y_inst[frame_mask]
    
    # Indices globaux pour mettre à jour labels_pred
    global_indices = np.where(frame_mask)[0]
    
    if t == 0:
        # Première frame : on lance HGP-Clusterer
        unique_classes = np.unique(Y_sem_frame)
        print(f"Classes sémantiques trouvées dans la frame 0 : {unique_classes}")
        
        for semantic_class in unique_classes:
            if semantic_class not in THINGS_CLASSES and SEMANTIC_MODE == "Oracle":
                continue
            
            class_mask = (Y_sem_frame == semantic_class)
            X_class = X_frame[class_mask]
            Y_inst_class = Y_inst_frame[class_mask]
            class_global_indices = global_indices[class_mask]
            CURRENT_GT_INSTANCES = Y_inst_class
            
            if len(X_class) < MIN_CLUSTER_SIZE:
                continue
                
            print(f"--- Traitement de la classe sémantique {semantic_class} ({len(X_class)} points) ---")
            
            # Alpine utilise la largeur (Width) comme seuil de distance
            prior = ALPINE_PRIORS.get(semantic_class, {"W": 1.0})
            class_threshold = prior["W"] * DBSCAN_FACTOR
            print(f"  -> Seuil HGP (Width prior) : {class_threshold}m")
            clusterer = HGPClusterer(
                K=K, min_cluster_size=MIN_CLUSTER_SIZE, min_samples=K+1,
                splitting=SPLITTING_REGISTRY.get(SPLITTING_MODE),
                expZ=EXP_Z,
                method=class_threshold, # Seuil dynamique selon la classe
                backend=BACKEND,
                cgal_root=os.environ.get("CGALDELAUNAY_ROOT"),
                verbose=False
            )
            
            t0 = time.time()
            try:
                class_labels_pred = clusterer.fit_predict(X_class)
                num_clusters = len(set(class_labels_pred)) - (1 if -1 in class_labels_pred else 0)
                print(f"  -> {num_clusters} clusters trouvés en {time.time()-t0:.2f}s.")
                
                valid_cluster_mask = class_labels_pred >= 0
                if np.any(valid_cluster_mask):
                    class_labels_pred[valid_cluster_mask] += global_instance_offset
                    labels_pred[class_global_indices] = class_labels_pred
                    
                    # --- Représentation des clusters (Volume, Classe, Vitesse) ---
                    unique_clusters_class = np.unique(class_labels_pred[valid_cluster_mask])
                    for cid in unique_clusters_class:
                        # On récupère les points 3D originaux pour calculer le volume
                        pts_3d = X_4d[class_global_indices][class_labels_pred == cid][:, :3]
                        dims = np.max(pts_3d, axis=0) - np.min(pts_3d, axis=0)
                        volume = np.prod(dims)
                        
                        # Vitesse (inconnue sur la frame 0)
                        vitesse = [0.0, 0.0, 0.0]
                        
                        clusters_info[cid] = {
                            "classe": semantic_class,
                            "volume": volume,
                            "vitesse": vitesse,
                            "num_points": len(pts_3d)
                        }
                        
                    global_instance_offset = np.max(class_labels_pred) + 1
            except Exception as e:
                print(f"  -> Erreur durant le clustering de la classe {semantic_class}: {e}")
                
        print("\nRésumé des clusters de la frame 0 :")
        for cid, info in clusters_info.items():
            print(f"  Cluster {cid:03d} | Classe: {info['classe']} | Volume: {info['volume']:.2f} m³ | Vitesse (init): {info['vitesse']} | Points: {info['num_points']}")

    else:
        # Pour les frames suivantes, pour l'instant on ne fait rien (on prépare juste le streaming)
        # TODO: Implémenter le suivi (tracking/association)
        pass

print(f"\nStreaming initial terminé. Total instances uniques détectées : {global_instance_offset}\n")


In [ ]:
# @title 5.1 Évaluation Officielle SemanticKITTI (PQ, SQ, RQ)
import os
import shutil
import numpy as np
import yaml

if X_clustering is not None and len(X_clustering) > 0:
    print("Préparation des fichiers pour l'évaluation officielle (semantic-kitti-api)...")
    
    eval_dir = "/content/eval_data"
    pred_dir = "/content/eval_predictions"
    seq_str = f"{SEQUENCE_TO_TEST:02d}"
    
    gt_labels_dir = os.path.join(eval_dir, "sequences", seq_str, "labels")
    pred_labels_dir = os.path.join(pred_dir, "sequences", seq_str, "predictions")
    
    # Nettoyage précédent éventuel
    shutil.rmtree(eval_dir, ignore_errors=True)
    shutil.rmtree(pred_dir, ignore_errors=True)
    
    os.makedirs(gt_labels_dir, exist_ok=True)
    os.makedirs(pred_labels_dir, exist_ok=True)
    
    # Création d'une configuration personnalisée pour n'évaluer que cette séquence
    custom_cfg_path = "/content/custom_eval_config.yaml"
    with open("/content/semantic-kitti-api/config/semantic-kitti.yaml", 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['split']['valid'] = [SEQUENCE_TO_TEST]
    with open(custom_cfg_path, 'w') as f:
        yaml.dump(cfg, f)
    
    # On reconstruit les labels prédits frame par frame
    actual_num_frames = len(loader) - START_FRAME if NUM_FRAMES == -1 else NUM_FRAMES
    for i in range(actual_num_frames):
        idx = START_FRAME + i
        if idx >= len(loader): break
        
        # Copie du GT
        gt_file = loader.label_files[idx]
        shutil.copy(gt_file, os.path.join(gt_labels_dir, os.path.basename(gt_file)))
        
        # Récupération des labels originaux pour garder la sémantique de fond
        s_raw, _ = loader.get_labels(idx)
        s_mapped = LABEL_MAP_ARRAY[s_raw]
        
        if SEMANTIC_MODE == "Oracle":
            mask = np.array([sem in THINGS_CLASSES for sem in s_mapped])
            frame_indices = np.where(mask)[0]
        else:
            frame_indices = np.arange(len(s_raw))
            
        # Initialisation de la prédiction
        pred_label = s_raw.astype(np.uint32)
        
        mask_time = (Time_idx == i)
        inst_preds = labels_pred[mask_time]
        
        valid_inst_mask = inst_preds >= 0
        valid_local_indices = frame_indices[valid_inst_mask]
        # Offset +1 car l'ID 0 est réservé au background
        valid_inst_ids = inst_preds[valid_inst_mask] + 1 
        
        pred_label[valid_local_indices] = (s_raw[valid_local_indices] & 0xFFFF) | (valid_inst_ids.astype(np.uint32) << 16)
        
        # Sauvegarde
        pred_filename = os.path.join(pred_labels_dir, os.path.basename(gt_file))
        pred_label.tofile(pred_filename)
        
    print("Fichiers de prédiction générés. Lancement de evaluate_panoptic.py...")
    # On lance le script officiel sur ce mini-dataset de test
    !python /content/semantic-kitti-api/evaluate_panoptic.py --dataset {eval_dir} --predictions {pred_dir} --split valid --data_cfg {custom_cfg_path} --output /content/eval_output
    if os.path.exists("/content/eval_output/scores.txt"):
        print("\n--- RÉSULTATS DE L'ÉVALUATION ---")
        with open("/content/eval_output/scores.txt", "r") as f:
            print(f.read())
else:
    print("Pas de données pour l'évaluation.")


In [ ]:
# @title 6.1 Visualisation 3D
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if X_4d is not None and len(X_4d) > 0:
    max_points = 100000
    num_points = len(X_4d)
    if num_points > max_points:
        print(f"Sous-échantillonnage de {num_points} à {max_points} points...")
        idx = np.random.choice(num_points, max_points, replace=False)
    else:
        idx = np.arange(num_points)
    
    X_v = X_4d[idx]
    L_v = labels_pred[idx]
    GT_v = Y_inst[idx]
    S_v = Y_sem[idx]
    THINGS_NAMES = {
        1: "car", 2: "bicycle", 3: "motorcycle", 4: "truck",
        5: "other-vehicle", 6: "person", 7: "bicyclist", 8: "motorcyclist"
    }
    
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=('Ground Truth', 'HGP Clustering Result')
    )
    
    # --- Gauche : Ground Truth ---
    u_gt = np.unique(GT_v)
    for l in u_gt:
        if l <= 0: continue # Ignorer le bruit/non-assigné
        m = GT_v == l
        # Récupération du nom de la classe
        class_name = THINGS_NAMES.get(S_v[m][0], "unknown")
        hover_text = [f"Class: {class_name}<br>GT Instance: {l}<br>Pred Instance: {L_v[m][i]}" for i in range(np.sum(m))]
        fig.add_trace(go.Scatter3d(
            x=X_v[m, 0], y=X_v[m, 1], z=X_v[m, 3],
            mode='markers', marker=dict(size=2), name=f'GT {l}',
            text=hover_text, hoverinfo='x+y+z+text+name'
        ), row=1, col=1)
    
    # --- Droite : Clustering Result ---
    u_l = np.unique(L_v)
    for l in u_l:
        if l == -1: continue # Ignorer le bruit
        m = L_v == l
        # Récupération du nom de la classe
        class_name = THINGS_NAMES.get(S_v[m][0], "unknown")
        hover_text = [f"Class: {class_name}<br>Pred Instance: {l}<br>GT Instance: {GT_v[m][i]}" for i in range(np.sum(m))]
        fig.add_trace(go.Scatter3d(
            x=X_v[m, 0], y=X_v[m, 1], z=X_v[m, 3],
            mode='markers', marker=dict(size=2), name=f'Pred {l}',
            text=hover_text, hoverinfo='x+y+z+text+name'
        ), row=1, col=2)
    
    fig.update_layout(
        title_text="Comparaison 4D (BEV: x, y, t) - GT vs Clustering",
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Time (t)'
        ),
        scene2=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Time (t)'
        ),
        showlegend=False
    )
    fig.show()
else:
    print("Pas de données à afficher.")
